# Selección de Variables NHANES

Este notebook tiene como objetivo cargar los datasets desde la capa `intermediate` de Kedro y aplicar un filtro de columnas basado en nuestra selección de variables determinantes para la esperanza de vida. Este es un paso previo interactivo antes de consolidar el proceso en el pipeline `data_processing`.

In [ ]:
%load_ext kedro.ipython

## 1. Definición de Variables Seleccionadas
Definiremos un diccionario que asocia cada dataset con la lista de columnas que deseamos retener. Usaremos la convención de Kedro para los nombres de los datasets en el catálogo.

In [ ]:
variables_seleccionadas = {
    'p_demo_intermediate': ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR'],
    'p_alq_intermediate': ['SEQN', 'ALQ121', 'ALQ130'],
    'p_bmx_intermediate': ['SEQN', 'BMXBMI', 'BMXWAIST'],
    'p_bpxo_intermediate': ['SEQN', 'BPXOSY1', 'BPXODI1'],
    'p_diq_intermediate': ['SEQN', 'DIQ010'],
    'p_ghb_intermediate': ['SEQN', 'LBXGH'],
    'p_hdl_intermediate': ['SEQN', 'LBDHDD'],
    'p_tchol_intermediate': ['SEQN', 'LBXTC'],
    'p_trigly_intermediate': ['SEQN', 'LBDLDL', 'LBXTR'],
    'p_mcq_intermediate': ['SEQN', 'MCQ220'],
    'p_paq_intermediate': ['SEQN', 'PAQ650', 'PAD680'],
    'p_slq_intermediate': ['SEQN', 'SLD012', 'SLQ030', 'SLQ040'],
    'p_smq_intermediate': ['SEQN', 'SMQ040', 'SMD057'],
    'p_biopro_intermediate': ['SEQN', 'LBXSCR', 'LBXSBU', 'LBXSATSI', 'LBXSASSI', 'LBXSAL']
}

# Las variables de la serie MCQ160 tienen sub-preguntas (MCQ160a, MCQ160b, etc.)
prefijos_adicionales = {
    'p_mcq_intermediate': ['MCQ160']
}

## 2. Función de Filtrado
Creamos la función que aplicará el filtro, la cual será la misma que se usará en los nodos del pipeline.

In [ ]:
import pandas as pd

def select_features(df: pd.DataFrame, exact_features: list, prefix_features: list = None) -> pd.DataFrame:
    """Selecciona columnas especificadas por su nombre exacto o por su prefijo."""
    cols_to_keep = []
    if prefix_features is None:
        prefix_features = []
        
    for col in df.columns:
        if col in exact_features:
            cols_to_keep.append(col)
        elif any(col.startswith(p) for p in prefix_features):
            cols_to_keep.append(col)
            
    return df[cols_to_keep]

## 3. Demostración del Filtrado
Cargamos algunos datasets desde el catálogo intermedio para verificar que las columnas sean exactamente las que buscamos.

In [ ]:
# Prueba 1: P_DEMO
dataset_name = 'p_demo_intermediate'
df_demo = catalog.load(dataset_name)
print(f"Columnas originales en {dataset_name}: {len(df_demo.columns)}")

df_demo_filtered = select_features(df_demo, variables_seleccionadas[dataset_name])
print(f"Columnas filtradas en {dataset_name}: {len(df_demo_filtered.columns)}")
display(df_demo_filtered.head())

In [ ]:
# Prueba 2: P_MCQ (comprobando el prefijo MCQ160)
dataset_name_mcq = 'p_mcq_intermediate'
df_mcq = catalog.load(dataset_name_mcq)
print(f"\nColumnas originales en {dataset_name_mcq}: {len(df_mcq.columns)}")

df_mcq_filtered = select_features(
    df_mcq, 
    exact_features=variables_seleccionadas.get(dataset_name_mcq, []), 
    prefix_features=prefijos_adicionales.get(dataset_name_mcq, [])
)
print(f"Columnas filtradas en {dataset_name_mcq}: {len(df_mcq_filtered.columns)}")
display(df_mcq_filtered.head())